# Colon

# GO-LR + NSC + TabICL

In [ ]:
# ============================================================
# GO-LR (fixed best) + NSC (fixed best) + TabICL
# Dataset: Colon (binary)
# Eval: 5x5 CV
# Reports:
#   - Accuracy mean ± std
#   - ROC-AUC mean ± std
#   - F1 mean ± std
#   - Runtime
# GPU: cuda:7
# ============================================================

import os
# ---- set CUDA before torch import ----
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
import gc
import time
import random
import inspect
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.utils.validation import check_X_y, check_array

from gotabpfn import GraphFeatureOrdering, pidf_segpca
PIDFSegPCA = pidf_segpca

from tabicl import TabICLClassifier
from tabicl.sklearn.preprocessing import (
    EnsembleGenerator,
    UniqueFeatureFilter,
    PreprocessingPipeline,
    CustomStandardScaler,
    OutlierRemover,
    TransformToNumerical,
)

# ============================================================
# Config
# ============================================================
SEED = 42
DATA_FILE = "coloncancer_encoded.csv"
TARGET_COL = "label"

BEST_PARAMS = {
    "go_metric": "euclidean",
    "go_num_clusters": 10,
    "go_refine_passes": 3,
    "go_direction_select": True,
    "nsc_segmentation": "equal_mass",
    "nsc_m_rule": "idf",
    "nsc_tau": 0.99,
    "nsc_gamma": 1.7570143129240916,
    "nsc_beta": 0.2244046472232107,
    "nsc_Mmin": 64,
    "nsc_Mmax": 384,
    "nsc_lmin": 16,
    "assume_standardized": False,
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# Seeds / utils
# ============================================================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()

def ensure_binary_contiguous(y):
    y = np.asarray(y).reshape(-1)
    y_enc = LabelEncoder().fit_transform(y.astype(str)).astype(np.int64)
    uniq = np.unique(y_enc)
    if len(uniq) != 2:
        raise ValueError(f"Expected binary labels, but found {len(uniq)} classes: {uniq}")
    return y_enc

def compute_deltas_adjacent_corr(X_tr: np.ndarray, Pi_star: list, eps: float = 1e-12) -> torch.Tensor:
    """
    delta[t] = 1 - |corr(feature_{Pi[t]}, feature_{Pi[t+1]})|
    Returns torch.Tensor shape (m-1,) on CPU.
    """
    X = torch.from_numpy(X_tr).float()
    perm = torch.tensor(Pi_star, dtype=torch.long)
    Xp = X[:, perm]
    Xc = Xp - Xp.mean(dim=0, keepdim=True)
    std = Xc.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
    Z = Xc / std
    corr = (Z[:, :-1] * Z[:, 1:]).mean(dim=0)
    return (1.0 - corr.abs()).cpu()

# ============================================================
# Patch TabICL internals
# ============================================================
def _set_feature_metadata(est, X):
    X_arr = X
    if hasattr(X, "values"):
        X_arr = X.values
    est.n_features_in_ = X_arr.shape[1]
    if hasattr(X, "columns"):
        est.feature_names_in_ = np.array(X.columns, dtype=object)

def _clf_validate_data(self,
                       X,
                       y=None,
                       dtype=None,
                       cast_to_ndarray=False,
                       skip_check_array=False,
                       **kwargs):
    if skip_check_array:
        _set_feature_metadata(self, X)
        return X, y

    if isinstance(X, pd.DataFrame):
        cols = X.columns
        X_out, y_out = check_X_y(X.values, y, accept_sparse=False, dtype=dtype)
        X_df = pd.DataFrame(X_out, columns=cols)
        _set_feature_metadata(self, X_df)
        return X_df, y_out

    X_out, y_out = check_X_y(X, y, accept_sparse=False, dtype=dtype)
    _set_feature_metadata(self, X_out)
    return X_out, y_out

TabICLClassifier._validate_data = _clf_validate_data

def _eg_validate_data(self,
                      X,
                      y=None,
                      dtype=None,
                      **kwargs):
    if isinstance(X, pd.DataFrame):
        cols = X.columns
        X_out, y_out = check_X_y(X.values, y, accept_sparse=False, dtype=dtype)
        X_df = pd.DataFrame(X_out, columns=cols)
        _set_feature_metadata(self, X_df)
        return X_df, y_out

    X_out, y_out = check_X_y(X, y, accept_sparse=False, dtype=dtype)
    _set_feature_metadata(self, X_out)
    return X_out, y_out

EnsembleGenerator._validate_data = _eg_validate_data

def _patched_uff_fit(self, X, y=None):
    X_arr = check_array(X, accept_sparse=False)

    self.n_features_in_ = X_arr.shape[1]
    if hasattr(X, "columns"):
        self.feature_names_in_ = np.array(X.columns, dtype=object)

    thr = getattr(self, "threshold", 1)

    if X_arr.shape[0] <= thr:
        self.features_to_keep_ = np.ones(self.n_features_in_, dtype=bool)
    else:
        self.features_to_keep_ = np.array(
            [len(np.unique(X_arr[:, i])) > thr for i in range(self.n_features_in_)]
        )

    self.n_features_out_ = int(self.features_to_keep_.sum())
    return self

def _patched_uff_transform(self, X):
    X_arr = check_array(X, accept_sparse=False)
    return X_arr[:, self.features_to_keep_]

UniqueFeatureFilter.fit = _patched_uff_fit
UniqueFeatureFilter.transform = _patched_uff_transform

def _generic_validate_data(self,
                           X,
                           y=None,
                           reset=True,
                           dtype=None,
                           **kwargs):
    if isinstance(X, pd.DataFrame):
        cols = X.columns
        X_out = check_array(X.values, accept_sparse=False, dtype=dtype)
        X_df = pd.DataFrame(X_out, columns=cols)
        _set_feature_metadata(self, X_df)
        return X_df

    X_out = check_array(X, accept_sparse=False, dtype=dtype)
    _set_feature_metadata(self, X_out)
    return X_out

PreprocessingPipeline._validate_data = _generic_validate_data
CustomStandardScaler._validate_data = _generic_validate_data
OutlierRemover._validate_data = _generic_validate_data

def _ttn_transform_identity(self, X):
    if isinstance(X, pd.DataFrame):
        arr = X.values

    elif isinstance(X, (list, tuple)):
        elems = [e for e in X if e is not None]
        if len(elems) == 0:
            raise ValueError("TransformToNumerical: got empty list/tuple")

        if len(elems) == 1 and hasattr(elems[0], "shape"):
            arr = np.asarray(elems[0])
        else:
            arrs = [np.asarray(e) for e in elems if hasattr(e, "shape")]
            if len(arrs) == 0:
                raise ValueError(f"TransformToNumerical: cannot interpret X={X!r}")
            n_samples = arrs[0].shape[0]
            arrs = [a.reshape(n_samples, -1) for a in arrs]
            arr = np.concatenate(arrs, axis=1)

    else:
        arr = np.asarray(X)

        if arr.ndim == 1 and arr.dtype == object:
            elems = [e for e in arr if e is not None]
            if len(elems) == 0:
                raise ValueError("TransformToNumerical: got empty object array")
            if len(elems) == 1 and hasattr(elems[0], "shape"):
                arr = np.asarray(elems[0])
            else:
                arrs = [np.asarray(e) for e in elems if hasattr(e, "shape")]
                n_samples = arrs[0].shape[0]
                arrs = [a.reshape(n_samples, -1) for a in arrs]
                arr = np.concatenate(arrs, axis=1)

    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    return arr

TransformToNumerical.transform = _ttn_transform_identity

# ============================================================
# TabICL constructor helper
# ============================================================
def make_tabicl_classifier(random_state):
    kwargs = {"random_state": random_state, "verbose": False}
    try:
        sig = inspect.signature(TabICLClassifier)
        if "device" in sig.parameters:
            kwargs["device"] = DEVICE
    except Exception:
        pass
    return TabICLClassifier(**kwargs)

# ============================================================
# Load Colon data
# ============================================================
seed_everything(SEED)

df = pd.read_csv(DATA_FILE)
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in {DATA_FILE}")

X_df = df.drop(columns=[TARGET_COL])
X_df = X_df.select_dtypes(include=[np.number]).copy()
X_df = X_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

y_all = ensure_binary_contiguous(df[TARGET_COL].to_numpy())

# global standardization to match your earlier pipeline
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df.values).astype(np.float32, copy=False)

X_all = np.asarray(X_scaled, dtype=np.float32, order="C")
y_all = np.asarray(y_all, dtype=np.int64)

print(f"[DATA] {DATA_FILE} | X={X_all.shape} | classes={np.unique(y_all)}")
print(f"[CUDA] available={torch.cuda.is_available()} | device_count={torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"[CUDA] using logical device: {DEVICE} (physical GPU 7 via CUDA_VISIBLE_DEVICES=7)")
    print(f"[CUDA] name: {torch.cuda.get_device_name(0)}")

# ============================================================
# GO-LR once on FULL X
# ============================================================
t0_total = time.time()

go = GraphFeatureOrdering(
    num_clusters=int(BEST_PARAMS["go_num_clusters"]),
    metric=BEST_PARAMS["go_metric"],
    refine=True,
    direction_select=bool(BEST_PARAMS["go_direction_select"]),
    refine_passes=int(BEST_PARAMS["go_refine_passes"]),
)

t0_go = time.time()
try:
    Pi_star, _, _, _ = go.fit(X_all, seed=SEED, deterministic=True, use_cpu_kmeans=False)
except RuntimeError:
    cleanup_cuda()
    Pi_star, _, _, _ = go.fit(X_all, seed=SEED, deterministic=True, use_cpu_kmeans=True)
t1_go = time.time()

print(f"[GO-LR] done in {t1_go - t0_go:.2f} sec | len(Pi_star)={len(Pi_star)}")

# ============================================================
# 5x5 CV with NSC -> TabICL
# ============================================================
cv_5x5 = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)

accs, aucs, f1s, fold_times = [], [], [], []

for fold_id, (tr_idx, va_idx) in enumerate(cv_5x5.split(X_all, y_all), start=1):
    t0_fold = time.time()

    X_tr = X_all[tr_idx]
    y_tr = y_all[tr_idx]
    X_va = X_all[va_idx]
    y_va = y_all[va_idx]

    # NSC on train only
    nsc = PIDFSegPCA(
        segmentation=BEST_PARAMS["nsc_segmentation"],
        l_min=int(BEST_PARAMS["nsc_lmin"]),
        m_rule=BEST_PARAMS["nsc_m_rule"],
        gamma=float(BEST_PARAMS["nsc_gamma"]),
        beta=float(BEST_PARAMS["nsc_beta"]),
        tau=float(BEST_PARAMS["nsc_tau"]),
        M_min=int(BEST_PARAMS["nsc_Mmin"]),
        M_max=int(BEST_PARAMS["nsc_Mmax"]),
        assume_standardized=bool(BEST_PARAMS["assume_standardized"]),
        device=DEVICE,
    )

    deltas = None
    if BEST_PARAMS["nsc_segmentation"] != "uniform":
        deltas = compute_deltas_adjacent_corr(X_tr, Pi_star)

    X_tr_t = torch.from_numpy(X_tr)
    X_va_t = torch.from_numpy(X_va)

    nsc.configure(
        Pi_star=Pi_star,
        X_train=X_tr_t,
        tau=float(BEST_PARAMS["nsc_tau"]),
        deltas=deltas,
    )

    Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy().astype(np.float32, copy=False)
    Z_va = nsc.compress(X_va_t, mode="flatten").cpu().numpy().astype(np.float32, copy=False)

    # make DataFrame for TabICL stability / feature names
    Z_tr_df = pd.DataFrame(Z_tr, columns=[f"z_{i}" for i in range(Z_tr.shape[1])])
    Z_va_df = pd.DataFrame(Z_va, columns=[f"z_{i}" for i in range(Z_va.shape[1])])

    clf = make_tabicl_classifier(SEED + fold_id)
    clf.fit(Z_tr_df, y_tr)

    y_pred = clf.predict(Z_va_df)
    y_proba = clf.predict_proba(Z_va_df)

    if hasattr(y_proba, "values"):
        y_proba = y_proba.values
    else:
        y_proba = np.asarray(y_proba)

    acc = float(accuracy_score(y_va, y_pred))
    auc = float(roc_auc_score(y_va, y_proba[:, 1]))
    f1  = float(f1_score(y_va, y_pred, average="binary"))

    accs.append(acc)
    aucs.append(auc)
    f1s.append(f1)

    t1_fold = time.time()
    fold_times.append(t1_fold - t0_fold)

    print(
        f"[Fold {fold_id:02d}/25] "
        f"ACC={acc:.6f} | AUC={auc:.6f} | F1={f1:.6f} | "
        f"time={t1_fold - t0_fold:.2f}s | Z={Z_tr.shape[1]}"
    )

    cleanup_cuda()

t1_total = time.time()

# ============================================================
# Final summary
# ============================================================
acc_mean, acc_std = float(np.mean(accs)), float(np.std(accs, ddof=1))
auc_mean, auc_std = float(np.mean(aucs)), float(np.std(aucs, ddof=1))
f1_mean,  f1_std  = float(np.mean(f1s)),  float(np.std(f1s, ddof=1))

total_runtime = t1_total - t0_total
mean_fold_time = float(np.mean(fold_times))

print("\n" + "=" * 64)
print("FINAL 5x5 CV RESULTS: GO-LR + NSC + TabICL (Colon, binary)")
print("=" * 64)
print(f"Accuracy : {acc_mean:.6f} ± {acc_std:.6f}")
print(f"ROC-AUC  : {auc_mean:.6f} ± {auc_std:.6f}")
print(f"F1       : {f1_mean:.6f} ± {f1_std:.6f}")
print(f"Runtime  : {total_runtime:.2f} sec ({total_runtime/60.0:.2f} min)")
print(f"Avg/fold : {mean_fold_time:.2f} sec")
print("=" * 64)

[DATA] coloncancer_encoded.csv | X=(62, 2000) | classes=[0 1]
[CUDA] available=True | device_count=1
[CUDA] using logical device: cuda (physical GPU 7 via CUDA_VISIBLE_DEVICES=7)
[CUDA] name: NVIDIA TITAN RTX
[GO-LR] done in 36.69 sec | len(Pi_star)=2000
[Fold 01/25] ACC=1.000000 | AUC=1.000000 | F1=1.000000 | time=16.59s | Z=64
[Fold 02/25] ACC=0.846154 | AUC=0.925000 | F1=0.800000 | time=14.86s | Z=64
[Fold 03/25] ACC=0.750000 | AUC=0.875000 | F1=0.666667 | time=15.95s | Z=64
[Fold 04/25] ACC=0.833333 | AUC=0.812500 | F1=0.750000 | time=13.47s | Z=64
[Fold 05/25] ACC=0.916667 | AUC=1.000000 | F1=0.888889 | time=17.72s | Z=64
[Fold 06/25] ACC=1.000000 | AUC=1.000000 | F1=1.000000 | time=11.90s | Z=64
[Fold 07/25] ACC=0.615385 | AUC=0.700000 | F1=0.444444 | time=10.73s | Z=64
[Fold 08/25] ACC=1.000000 | AUC=1.000000 | F1=1.000000 | time=9.28s | Z=64
[Fold 09/25] ACC=0.916667 | AUC=0.906250 | F1=0.888889 | time=9.03s | Z=64
[Fold 10/25] ACC=0.833333 | AUC=0.937500 | F1=0.750000 | time=1